In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
from openaq_anomaly_prediction.config import Configuration as cfg
from openaq_anomaly_prediction.core.seoul_map import SeoulMap

import requests

import plotly.express as px


### Create Parquet

In [2]:
anomalies_path = os.path.join(cfg.DATA_EXPORT_PATH, "anomalies_top_100pct_contributors.parquet")

anomalies_df = pd.read_parquet(anomalies_path)
anomalies_df.loc[:, "contribution_pct"] = anomalies_df["contribution_pct"] / 100
anomalies_df.loc[:, "cumulative_contribution_pct"] = anomalies_df["cumulative_contribution_pct"] / 100

anomalies_df = anomalies_df.drop(columns=["feature", "error"])
anomalies_df = anomalies_df.rename(columns={"timestamp": "ds"})
anomalies_df = anomalies_df[["ds", "location_id", "rank", "pollutant", "contribution_pct", "cumulative_contribution_pct"]]

In [3]:
big_path = os.path.join(cfg.DATA_EXPORT_PATH, "all_best_predictions_v1.parquet")
big_df = pd.read_parquet(big_path)
locations_df = big_df.groupby("location_id")[["oaq_latitude", "oaq_longitude"]].first().reset_index()
locations_df.head()

,location_id,oaq_latitude,oaq_longitude
0,2622586,37.580167,127.044856
1,2622594,37.539109,127.214608
2,2622600,37.427400,126.746100
3,2622606,37.635600,127.216200
4,2622608,37.516083,127.019694


In [4]:
anomalies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17808 entries, 0 to 17807
Data columns (total 6 columns):
 #   Column                       Non-Null Count  Dtype              
---  ------                       --------------  -----              
 0   ds                           17808 non-null  datetime64[ns, UTC]
 1   location_id                  17808 non-null  object             
 2   rank                         17808 non-null  int64              
 3   pollutant                    17808 non-null  object             
 4   contribution_pct             17808 non-null  float64            
 5   cumulative_contribution_pct  17808 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(2), int64(1), object(2)
memory usage: 834.9+ KB


In [5]:
new_df = anomalies_df.merge(locations_df, on="location_id", how="left")
new_df.head(50)

,ds,location_id,rank,pollutant,contribution_pct,cumulative_contribution_pct,oaq_latitude,oaq_longitude
0,2025-11-07 09:00:00+00:00,2623345,1,no2_ppm,0.013704,0.013704,37.443100,126.788400
1,2025-11-07 09:00:00+00:00,2623720,2,no2_ppm,0.013458,0.027162,37.404839,126.697158
2,2025-11-07 09:00:00+00:00,2623508,3,no2_ppm,0.013449,0.040610,37.619335,126.716748
3,2025-11-07 09:00:00+00:00,2623811,4,no2_ppm,0.013157,0.053767,37.474899,126.898657
4,2025-11-07 09:00:00+00:00,2623191,5,no2_ppm,0.012945,0.066712,37.606667,127.027264
5,2025-11-07 09:00:00+00:00,2623214,6,no2_ppm,0.012739,0.079451,37.654509,126.775557
6,2025-11-07 09:00:00+00:00,2623194,7,no2_ppm,0.012584,0.092035,37.489495,126.982489
7,2025-11-07 09:00:00+00:00,2622718,8,no2_ppm,0.012546,0.104581,37.594100,126.698600
8,2025-11-07 09:00:00+00:00,2623679,9,no2_ppm,0.012336,0.116917,37.562821,126.826071
9,2025-11-07 09:00:00+00:00,2623430,10,no2_ppm,0.010971,0.127888,37.617315,127.075120


In [65]:
new_df.to_parquet(os.path.join(cfg.DATA_EXPORT_PATH, "anomalies_with_coordinates.parquet"), index=False)

### Load Parquet

In [7]:
new_df = pd.read_parquet(os.path.join(cfg.DATA_EXPORT_PATH, "anomalies_with_coordinates.parquet"))

In [21]:
anomalies_df["timestamp"].nunique()

56

In [54]:
new_df.head(2)

,ds,location_id,rank,pollutant,contribution_pct,cumulative_contribution_pct,oaq_latitude,oaq_longitude
0,2025-11-07 09:00:00+00:00,2623345,1,no2_ppm,0.013704,0.013704,37.443100,126.788400
1,2025-11-07 09:00:00+00:00,2623720,2,no2_ppm,0.013458,0.027162,37.404839,126.697158


In [ ]:
anomalies_df.groupby("timestamp")["location_id"].count()

In [ ]:
anomalies_df.head(20)

### Anomalies Map

In [ ]:
# seoul_geojson_url = "https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2018/json/skorea-provinces-2018-geo.json"
# seoul_geojson_features = requests.get(seoul_geojson_url).json()["features"]

# seoul_geojson_districts_url = "https://raw.githubusercontent.com/southkorea/seoul-maps/refs/heads/master/juso/2015/json/seoul_municipalities_geo_simple.json"
# seoul_geojson_districts = requests.get(seoul_geojson_districts_url).json()

# korea_municipalities_url = "https://raw.githubusercontent.com/southkorea/southkorea-maps/master/kostat/2018/json/skorea-municipalities-2018-geo.json"
# resp = requests.get(korea_municipalities_url)

In [32]:
seoul_map = SeoulMap()

In [64]:
single_anomaly = new_df[new_df["ds"] == "2025-11-07 09:00:00+00:00"]
single_anomaly = single_anomaly.groupby("location_id")["contribution_pct"].sum().reset_index().merge(
    new_df[["location_id", "oaq_latitude", "oaq_longitude"]].drop_duplicates(),
    on="location_id",
    how="left",
)

display(single_anomaly.head())

min_threshold = 0
max_threshold = single_anomaly["contribution_pct"].max()
if max_threshold < 0.05:
    max_threshold = 0.05


fig = seoul_map.scatter_map(
    single_anomaly,
    lat_column="oaq_latitude",
    lon_column="oaq_longitude",
    # title="SEOUL - Scatter Map",
    date_col="ds",
    label="Contribution %",
    color_column="contribution_pct",
    size_column="contribution_pct",
    size_max=64,
    exponential=True,
    coloraxes_min=min_threshold,
    coloraxes_max=max_threshold,
    # coloraxes_min=0,
    # coloraxes_max=current_threshold,
    map_style="basic",
    # map_style="satellite",
    hover_name="location_id",
    hover_data={
        "location_id": True,
        # "location_municipality": True,
        # "nan_pct": ":.2%",
        "oaq_latitude": False,
        "oaq_longitude": False,
        "contribution_pct": ":.2%",
        # "location_perfect_hours": True,
        # "location_type": True,
    },
    map_layers=True,
    map_layers_glowing_border=False,
    map_layers_region="seoul",
)

fig.show()

,location_id,contribution_pct,oaq_latitude,oaq_longitude
0,2622600,0.012066,37.427400,126.746100
1,2622609,0.022800,37.668300,126.756400
2,2622689,0.022952,37.657415,127.067876
3,2622718,0.033379,37.594100,126.698600
4,2622724,0.015033,37.433900,127.129300


### Bar chart

In [58]:
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17808 entries, 0 to 17807
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype              
---  ------                       --------------  -----              
 0   ds                           17808 non-null  datetime64[ns, UTC]
 1   location_id                  17808 non-null  object             
 2   rank                         17808 non-null  int64              
 3   pollutant                    17808 non-null  object             
 4   contribution_pct             17808 non-null  float64            
 5   cumulative_contribution_pct  17808 non-null  float64            
 6   oaq_latitude                 17808 non-null  float64            
 7   oaq_longitude                17808 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(4), int64(1), object(2)
memory usage: 1.1+ MB


In [22]:
new_df.head()

,ds,location_id,rank,pollutant,contribution_pct,cumulative_contribution_pct,oaq_latitude,oaq_longitude
0,2025-11-07 09:00:00+00:00,2623345,1,no2_ppm,0.013704,0.013704,37.443100,126.788400
1,2025-11-07 09:00:00+00:00,2623720,2,no2_ppm,0.013458,0.027162,37.404839,126.697158
2,2025-11-07 09:00:00+00:00,2623508,3,no2_ppm,0.013449,0.040610,37.619335,126.716748
3,2025-11-07 09:00:00+00:00,2623811,4,no2_ppm,0.013157,0.053767,37.474899,126.898657
4,2025-11-07 09:00:00+00:00,2623191,5,no2_ppm,0.012945,0.066712,37.606667,127.027264


In [29]:
single_anomaly_per_location = new_df[new_df["ds"] == "2025-11-07 09:00:00+00:00"]
single_anomaly_per_location = (
    single_anomaly_per_location.groupby("location_id")["contribution_pct"]
    .sum()
    .reset_index()
    .merge(
        new_df[
            ["location_id", "oaq_latitude", "oaq_longitude"]
        ].drop_duplicates(),
        on="location_id",
        how="left",
    )
)
single_anomaly_per_location.head()

,location_id,contribution_pct,oaq_latitude,oaq_longitude
0,2622600,0.012066,37.427400,126.746100
1,2622609,0.022800,37.668300,126.756400
2,2622689,0.022952,37.657415,127.067876
3,2622718,0.033379,37.594100,126.698600
4,2622724,0.015033,37.433900,127.129300


In [28]:
single_anomaly_per_pollutant = new_df[new_df["ds"] == "2025-11-07 09:00:00+00:00"]
single_anomaly_per_pollutant = single_anomaly_per_pollutant.groupby("pollutant")["contribution_pct"].sum().reset_index()
display(single_anomaly_per_pollutant)

,pollutant,contribution_pct
0,co_ppm,0.167434
1,no2_ppm,0.333818
2,o3_ppm,0.156453
3,pm10_ugm3,0.074237
4,pm25_ugm3,0.130164
5,so2_ppm,0.137893


In [57]:
POLLUTANT_NAMES = {
    "pm25_ugm3": "PM2.5 µg/m³",
    "pm10_ugm3": "PM10 µg/m³",
    "o3_ppm": "O3 ppm",
    "no2_ppm": "NO2 ppm",
    "so2_ppm": "SO2 ppm",
    "co_ppm": "CO ppm",
}


KOREAN_FLAG_CS = [
    # # TIER 1: Deep Ink (0% - 25%)
    # [0.0, "rgba(20, 20, 20, 0.1)"],
    # [0.25, "rgba(20, 20, 20, 0.2)"],
    # TIER 1.5: Deep Ink (0% - 25%)
    [0.0, "rgba(60, 60, 60, 0.4)"],
    [0.25, "rgba(60, 60, 60, 0.5)"],
    # # TIER 2: Official Blue (25% - 50%)
    # [0.25, "rgba(0, 71, 160, 0.6)"],
    # [0.5, "rgba(0, 71, 160, 0.7)"],
    # TIER 2.5: Official Blue (25% - 50%)
    [0.25, "rgba(0, 71, 160, 0.5)"],
    [0.5, "rgba(0, 71, 160, 0.6)"],
    # TIER 3: Royal Yellow (50% - 75%)
    [0.5, "rgba(255, 209, 103, 0.8)"],
    [0.75, "rgba(255, 209, 103, 0.8)"],
    # TIER 4: Cinnabar Red (75% - 100%)
    [0.75, "rgba(205, 46, 58, 0.9)"],
    [1.0, "rgba(205, 46, 58, 0.9)"],
]


# # FULL MANUAL
# fig = go.Figure()
# fig.add_trace(go.Bar(
#     x=months,
#     y=[20, 14, 25, 16, 18, 22, 19, 15, 12, 16, 14, 17],
#     name='Primary Product',
#     marker_color='indianred'
# ))
# fig.add_trace(go.Bar(
#     x=months,
#     y=[19, 14, 22, 14, 16, 19, 15, 14, 10, 12, 12, 16],
#     name='Secondary Product',
#     marker_color='lightsalmon'
# ))

# ## FULL MANUAL: ROUNDED CORNERS
# fig = go.Figure(
#     data=[
#         go.Bar(x=df.nation, y=df.gold, name="Gold"),
#         go.Bar(x=df.nation, y=df.silver, name="Silver"),
#         go.Bar(x=df.nation, y=df.bronze, name="Bronze"),
#     ],
#     layout=dict(
#         barcornerradius=15,
#     ),
# )

# # POSSIBILITIES OF CUSTOMIZATION
# fig.update_traces(textfont_size=12, textangle=0, textposition="outside", cliponaxis=False)
# fig.update_traces(texttemplate='%{text:.2s}', textposition='outside')
# fig.update_layout(uniformtext_minsize=8, uniformtext_mode='hide')
# fig.update_layout(barmode='group', xaxis_tickangle=-45)

fig = px.bar(
    single_anomaly_per_pollutant.sort_values("contribution_pct", ascending=True),
    y="pollutant",
    x="contribution_pct",
    orientation='h',
    title="Anomaly Contribution by Pollutant",
    labels={"contribution_pct": "Contribution %", "pollutant": "Pollutant"},
    text_auto=".2%",  # .2s
    color="contribution_pct",
    color_continuous_scale=KOREAN_FLAG_CS,
    range_color=[0, single_anomaly_per_pollutant["contribution_pct"].max()],
    opacity=1,
)

fig.update_traces(
    marker_cornerradius=10  # Adjust this value for more or less rounding
)

original_values = list(POLLUTANT_NAMES.keys())
new_labels = [POLLUTANT_NAMES[k]+ "   " for k in original_values]

fig.update_yaxes(
    tickmode="array",
    tickvals=original_values,
    ticktext=new_labels,
)

# fig.update_yaxes(
#     labelalias=POLLUTANT_NAMES,
#     ticksuffix="   ",  # Adds 3 spaces of padding between labels and bars
# )

fig.update_traces(
    texttemplate='%{x:.2%}&nbsp;&nbsp;&nbsp;', # Adds two spaces after the number
    textposition="inside",
    insidetextanchor="end", # Keep it at the tip
    # You can also add font styling specifically for the inside text
    insidetextfont=dict(color="white", size=14, weight="bold") 
)

fig.update_layout(
    title=dict(
        text=f"<b>Anomalies % par type de capteur</b>",
        x=0.0,  # 0 is far left, 1 is far right
        y=0.96,  # 1 is very top, 0 is very bottom
        xanchor="left",  # Anchor the text's left side to the x-coordinate
        yanchor="top",  # Anchor the text's top side to the y-coordinate
        font=dict(size=20, family="Google Sans, sans-serif"),
    ),
    # height=400,
    margin=dict(l=0, t=36, r=0, b=0),
    plot_bgcolor="white",  # Clean background color
    font=dict(size=16, family="Google Sans, sans-serif"),  # Consistent font
    dragmode=False,
    coloraxis_showscale=False,
    showlegend=False,  # Ensure legend is on so we see "Prediction"
    legend=dict(
        orientation="h",  # Horizontal legend at bottom/top
        yanchor="bottom",
        y=1.06,
        xanchor="right",
        x=1,
    ),
    xaxis=dict(
        showgrid=False,
        gridcolor="rgba(200, 200, 200, 0.5)",  # Light gray grid
        gridwidth=1,
        fixedrange=True,
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="rgba(200, 200, 200, 0.5)",
        gridwidth=1,
        griddash="dot",  # solid, dot, dash, longdash, dashdot
        fixedrange=True,
    ),
)            
            
            
fig.show()